In [3]:
!pip install keras-facenet

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
    --------------------------------------- 0.0/1.9 MB 960.0 kB/s eta 0:00:02
   ------------- -------------------------- 0.6/1.9 MB 6.6 MB/s eta 0:00:01
   --------------------------------- ------ 1.6/1.9 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 1.9/1.9 MB 11.0 MB/s eta 0:00:00
Using cached joblib-1.4.2-py3-none-any.whl (301 kB)
   ---------------------------------------- 0.0/99.9 kB ? eta -:--:--
   ---------------------------------------- 99.9/99.9 kB ? eta 0:00:00
  Created wheel for keras-facenet: filename=keras_facenet-0.3.2-py3-none-any.whl size=10387 sha256=84c1f5184f28924127df33bb6e15d07a0e3d5bd061793ddfaf22cea02fb87538
  Stored in directory: c:\users\guill\appdata\local\pip\cache\wheels\99\94\dd\cb1a65a7440ba6d508bd243

## Load Libraries

In [1]:
import tensorflow as tf
from keras_facenet import FaceNet
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Input, Dense
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from mtcnn import MTCNN

## Load Data

In [2]:
# Load data paths
aug_selfies = r"G:\Mi Unidad\6 SEMESTER\3569_MCS5713_Deep Learning\Deep Learning Project - Facial Recognition\Data\archive (1)\selfies_data\training\augmented"
test_aug_selfies = r"G:\Mi Unidad\6 SEMESTER\3569_MCS5713_Deep Learning\Deep Learning Project - Facial Recognition\Data\archive (1)\selfies_data\testing\augmented"
full_class_ds = r"G:\Mi Unidad\6 SEMESTER\3569_MCS5713_Deep Learning\Deep Learning Project - Facial Recognition\Data\archive (1)\Augmented_Images"

In [3]:
# Define parameters

IMG_SIZE = (160, 160) #FaceNet expects (160,160)
BATCH_SIZE = 32
SEED = 42

In [4]:
train = tf.keras.utils.image_dataset_from_directory(
    full_class_ds,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

Found 210 files belonging to 21 classes.


In [5]:
class_names = train.class_names
print("Classes:", class_names)

Classes: ['Ajith Kumar Linga', 'Akram Ghaffari Gharebhagh', 'Alejandro Rollan', 'Andressa Braga Vieira Rodrigues', 'Bhargava Sai Gudikandula', 'Bhavesh Krishnaram Bhavesh', 'Guillermo Garcia de Celis Rodriguez', 'Jagadeeshwar Reddy Vanam', 'Kevin Truong', 'Lucas Ribeiro', 'Mitch Pleune', 'Mohammad Mezaael', 'Pavan Kumar Reddy Alavala', 'Shalini Paka', 'Siddarth Vuppula', 'Siranjeev Venkateswaran', 'Sujith Yamsani', 'Udaya Durga Purna Abhiram Devalla', 'Urvashi Rakholiya', 'Urvashiben Thakkar', 'Yash Kandu']


## Create Embedder and Embed Images

### Create the Embedder

In [6]:
# Build the embedder model
def create_embedder():    
    model = FaceNet()
    return model

In [7]:
embedder = create_embedder()

### Compute Embeddings for the database

In [34]:
embeddings_db = {}

# Iterate through each person's folder
for person_name in os.listdir(full_class_ds):
    person_folder = os.path.join(full_class_ds, person_name)
    if not os.path.isdir(person_folder):
        continue

    embeddings = []

    # Iterate through each image in the person's folder
    for image_name in os.listdir(person_folder):
        image_path = os.path.join(person_folder, image_name)
        image = cv2.imread(image_path)
        if image is None:
            continue

        # Convert BGR (OpenCV default) to RGB
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Detect faces and extract embeddings
        detections = embedder.extract(image_rgb, threshold=0.95)

        # If a face is detected, append its embedding
        if detections:
            embeddings.append(detections[0]['embedding'])

    # If embeddings were found for the person, compute the average embedding
    if embeddings:
        mean_embedding = np.mean(embeddings, axis=0)
        embeddings_db[person_name] = mean_embedding

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━

In [35]:
# Average multiple embeddings
final_embeddings_db = {name: vectors.flatten() for name, vectors in embeddings_db.items()}

# Save
np.save(r"G:\Mi Unidad\6 SEMESTER\3569_MCS5713_Deep Learning\Deep Learning Project - Facial Recognition\models\Embeddings\full_class_embeddings_db.npy", final_embeddings_db)

---

## Testing Embedder

In [36]:
# Set a similarity threshold
THRESHOLD = 0.5

# Initialize counters
correct = 0
total = 0

# Path to the test dataset
test_dataset_path = test_aug_selfies

# Iterate through each person's folder
for person_name in os.listdir(test_dataset_path):
    person_folder = os.path.join(test_dataset_path, person_name)
    if not os.path.isdir(person_folder):
        continue

    # Iterate through each image in the person's folder
    for image_name in os.listdir(person_folder):
        image_path = os.path.join(person_folder, image_name)
        image = cv2.imread(image_path)
        if image is None:
            continue

        # Convert BGR to RGB
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Detect faces and extract embeddings
        detections = embedder.extract(image_rgb, threshold=0.95)

        if detections:
            # Assume the first detected face is the target
            embedding = detections[0]['embedding']

            # Initialize variables to store the best match
            best_match = "Unknown"
            best_score = -1

            # Compare the embedding with each in the database
            for name, db_embedding in embeddings_db.items():
                score = cosine_similarity(embedding, db_embedding)
                if score > best_score:
                    best_score = score
                    best_match = name

            # Check if the best match exceeds the threshold
            if best_score >= THRESHOLD:
                predicted_name = best_match
            else:
                predicted_name = "Unknown"

            # Update counters
            total += 1
            if predicted_name == person_name:
                correct += 1

# Calculate accuracy
accuracy = correct / total if total > 0 else 0
print(f"Accuracy: {accuracy * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━

---

## Detection Application

### Create Face Detector

In [8]:
detector = MTCNN()

### Define Similarity Check

In [9]:
def cosine_similarity(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return np.dot(a, b)

### Create Real-Time Detection Application

In [10]:
# Load known embeddings
embeddings_db = np.load(r"G:\Mi Unidad\6 SEMESTER\3569_MCS5713_Deep Learning\Deep Learning Project - Facial Recognition\models\Embeddings\full_class_embeddings_db.npy", allow_pickle=True).item()

THRESHOLD = 0.5  # Cosine distance threshold (lower = more strict)

In [11]:
embedder = create_embedder()

In [14]:
# List of classmates
classmates = class_names

# Initialize attendance dictionary
attendance = {name: False for name in classmates}

# Start the webcam
cap = cv2.VideoCapture(0)

print("Starting webcam... Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert BGR to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detect faces and extract embeddings
    detections = embedder.extract(frame_rgb, threshold=0.95)

    for detection in detections:
        x, y, w, h = detection['box']
        embedding = detection['embedding']

        # Initialize variables to store the best match
        best_match = "Unknown"
        best_score = -1

        # Compare the embedding with each in the database
        for name, db_embedding in embeddings_db.items():
            score = cosine_similarity(embedding.flatten(), db_embedding)
            if score > best_score:
                best_score = score
                best_match = name

        # If the best score is below the threshold, label as Unknown
        if best_score < THRESHOLD:
            best_match = "Unknown"

        # If a valid classmate is recognized, mark attendance
        if best_match in classmates:
            attendance[best_match] = True  # Important: set to True when detected!

        # Draw a rectangle around the face
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Display the name and similarity score
        cv2.putText(frame, f"{best_match} ({best_score:.2f})", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # Show the frame
    cv2.imshow('Face Recognition', frame)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()

# --- Final Attendance Report ---
print("\nAttendance List:")
for name, present in attendance.items():
    status = "Present" if present else "Absent"
    print(f"{name}: {status}")

Starting webcam... Press 'q' to quit.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step

Attendance List:
Ajith Kumar Linga: Absent
Akram Ghaffari Gharebhagh: Absent
Alejandro Rollan: Absent
Andressa Braga Vieira Rodrigues: Absent
Bhargava Sai Gudikandula: Absent
Bhavesh Krishnaram Bhavesh: Absent
Guillermo Garcia de Celis Rodriguez: Present
Jagadeeshwar Reddy Vanam: Absent
Kevin Truong: Absent
Lucas Ribeiro: Absent
Mitch Pleune: Absent
Mohammad Mezaael: Absent
Pavan Kumar Reddy Alavala: Absent
Shalini Paka: Absent
Siddarth Vuppula: Absent
Siranjeev 